In [0]:
%sql
CREATE OR REPLACE VIEW users.fernando_vasquez.coffee_metric_view
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.2

source: users.fernando_vasquez.fct_coffee_shop_transactions

joins:
  - name: dim_coffee_shop_stores
    source: users.fernando_vasquez.dim_coffee_shop_stores
    "on": source.store_id = dim_coffee_shop_stores.store_id
    joins:
      - name: fct_coffee_shop_stores_reviews
        source: users.fernando_vasquez.fct_coffee_shop_stores_reviews
        "on": dim_coffee_shop_stores.store_id = fct_coffee_shop_stores_reviews.store_id
  - name: dim_coffee_products
    source: users.fernando_vasquez.dim_coffee_products
    "on": source.product_iD = dim_coffee_products.product_id

dimensions:
  - name: product
    expr: source.product_iD
    comment: Unique identifier for the coffee product
    display_name: Product ID
  - name: Product detail
    expr: dim_coffee_products.product_detail
    comment: Full descriptive name and details of the coffee product
    display_name: Product Detail
  - name: store_name
    expr: dim_coffee_shop_stores.store_name
    comment: Name of the coffee shop store where the transaction occurred
    display_name: Store Name
  - name: month_day
    expr: "make_date(year(transaction_date), month(transaction_date), day(transaction_date))"
    comment: Date of the transaction derived from the transaction timestamp
    display_name: Transaction Date
  - name: hour_of_day
    expr: HOUR(transaction_time)
    comment: Hour of the day (0-23) when the transaction took place
    display_name: Hour of Day

measures:
  - name: total_transactions
    expr: SUM(transaction_id)
    comment: Sum of all transaction IDs, representing overall transaction volume
    display_name: Total Transactions
  - name: total_sales
    expr: "ROUND(sum(transation_total),2)"
    comment: Total revenue from all transactions, rounded to two decimal places
    display_name: Total Sales
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
  - name: total_quantity_sold
    expr: SUM(transaction_qty)
    comment: Total number of items sold across all transactions
    display_name: Total Quantity Sold
$$;

result
